# Chess with python-chess

**python-chess** is a Python library that knows every rule of chess. It can set up a board, check whether a move is allowed, spot check and checkmate, and draw the board as a picture. Chess websites and chess engines use libraries like this so they don't have to code the rules themselves.

In this notebook you'll learn to:

- create a board and look at its squares and pieces
- make moves, take them back, and list every legal move
- describe a whole position in one line of text (FEN)
- tell when a game is over, and why
- write two simple chess-playing programs and make them play each other

You don't need to be good at chess. If you know how the pieces move, you're ready.

# The board

`chess.Board()` creates a board in the starting position. When a board is the last line of a cell, Codetto draws it as a picture.

`print(board)` gives a text version in the **stdout** tab: capital letters are White's pieces, lower-case letters are Black's, and dots are empty squares. `K` is the king, `Q` the queen, `R` a rook, `B` a bishop, `N` a knight (`K` was already taken) and `P` a pawn.

In [ ]:
#@alt Chess board in the starting position
import chess

board = chess.Board()
print(board)
board

## Squares and pieces

Every square has a name made of a column letter (`a` to `h`) and a row number (`1` to `8`). White starts on rows 1 and 2, Black on rows 7 and 8.

python-chess has a constant for every square, such as `chess.E1`. `board.piece_at(square)` tells you which piece is on a square, or gives `None` if it's empty.

In [ ]:
for square in [chess.E1, chess.D8, chess.A2, chess.E4]:
  piece = board.piece_at(square)
  name = chess.square_name(square)
  if piece is None:
    print(name, "is empty")
  else:
    colour = "White" if piece.color == chess.WHITE else "Black"
    print(name, "has a", colour, chess.piece_name(piece.piece_type))

# Making moves

`board.push_san("e4")` makes a move written in **SAN** (Standard Algebraic Notation), the notation used in chess books: `e4` moves a pawn to e4, `Nf3` moves a knight to f3, and `O-O` castles. The board keeps track of whose turn it is, so you push the moves in order.

When the board is drawn, the move that was just played is highlighted.

In [ ]:
#@alt Chess board after 1. e4 e5 2. Nf3, with the knight's move highlighted
board = chess.Board()
board.push_san("e4")
board.push_san("e5")
board.push_san("Nf3")

print("Moves so far:", len(board.move_stack))
print("White to move?", board.turn == chess.WHITE)
board

Each move is a `chess.Move` object. Besides SAN, a move can be written in **UCI** notation, which gives just the starting and ending squares, such as `g1f3`. Chess engines use UCI to talk to other programs.

`board.pop()` takes back the last move and returns it. A program can use it to try a move and then undo it.

In [ ]:
#@alt Chess board after 1. e4 e5, with White to move
last = board.pop()
print("Took back:", last.uci())
print("Moves so far:", len(board.move_stack))
print("White to move?", board.turn == chess.WHITE)
board

# Legal moves

`board.legal_moves` holds every move the player whose turn it is may make. python-chess applies all the rules for you, including check, castling, en passant and promotion. `board.san(move)` turns a move into SAN.

In [ ]:
board = chess.Board()
moves = list(board.legal_moves)

print("White has", len(moves), "legal moves:")
print(", ".join(board.san(move) for move in moves))

If you push a move that isn't allowed, python-chess raises a `ValueError` and leaves the board unchanged. You can also check first with `board.is_legal()`.

In [ ]:
move = chess.Move.from_uci("e2e5")
print("Is e2e5 legal?", board.is_legal(move))

try:
  board.push_san("Ke2")
except ValueError as error:
  print("Not allowed:", error)

### Try it yourself: where can a piece go?

`board.attacks(square)` gives the squares the piece on `square` can reach. The cell below puts one piece on an empty board and marks every square it can reach. Pick a piece and a square: the cell runs again whenever a field changes.

To mark the squares, it draws the board with `chess.svg.board()`, which can add marked squares, arrows and more to the picture. `display()` from `IPython.display` shows that picture, just as it would in any Jupyter notebook.

(Pawns are left out because they capture differently from the way they move.)

In [ ]:
#@title Where can it go?
#@alt Chess board with one white piece and every square it can reach marked
PIECE = "Knight" #@param ["Knight", "Bishop", "Rook", "Queen", "King"]
SQUARE = "d4" #@param ["a1", "d4", "e5", "b7", "g2", "h8"]

import chess
import chess.svg
from IPython.display import display

piece_types = {
  "Knight": chess.KNIGHT,
  "Bishop": chess.BISHOP,
  "Rook": chess.ROOK,
  "Queen": chess.QUEEN,
  "King": chess.KING,
}

empty = chess.Board(None)  # None means a board with no pieces at all
square = chess.parse_square(SQUARE)
empty.set_piece_at(square, chess.Piece(piece_types[PIECE], chess.WHITE))

reach = empty.attacks(square)
print(f"A {PIECE.lower()} on {SQUARE} can reach {len(reach)} squares")
display(chess.svg.board(empty, squares=reach, size=350))

# Positions as text: FEN

A whole position, including every piece, whose turn it is and who can still castle, fits in one line of text called **FEN** (Forsyth–Edwards Notation). `board.fen()` gives a board's FEN, and `chess.Board(fen)` sets up any position you like. Chess puzzle websites store their puzzles this way.

In [ ]:
print(chess.Board().fen())

The part before the first space describes the rows from 8 down to 1, separated by slashes. Letters are pieces and numbers count empty squares, so `8` is an empty row. After it, `w` means White to move and `KQkq` lists who can still castle.

Here's a puzzle: White to move and checkmate in one. Can you find the move before you run the next cell?

In [ ]:
#@alt Chess puzzle: White to move and checkmate in one
puzzle = chess.Board("r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4")
puzzle

Instead of solving it yourself, let the computer try every legal move. For each one, it pushes the move, checks `is_checkmate()`, then pops the move to put the board back.

This **try, check, undo** pattern is how chess programs search for good moves.

In [ ]:
for move in list(puzzle.legal_moves):
  san = puzzle.san(move)  # name the move before playing it
  puzzle.push(move)
  if puzzle.is_checkmate():
    print("Checkmate with", san)
  puzzle.pop()

# Is the game over?

python-chess can tell you when a game has ended, and why:

- `is_check()`: the player to move is in check
- `is_checkmate()`: in check with no legal moves, so that player has lost
- `is_stalemate()`: not in check, but no legal moves, so the game is a draw
- `is_game_over()`: the game has ended for any reason, including draws such as too few pieces left to checkmate
- `result()`: the score, `"1-0"` (White won), `"0-1"` (Black won) or `"1/2-1/2"` (a draw)

`board.copy()` makes a separate board, so playing the answer below leaves `puzzle` as it was.

In [ ]:
#@alt Chess board where Black, to move, is in stalemate
finished = puzzle.copy()
finished.push_san("Qxf7#")
print("Check?", finished.is_check())
print("Checkmate?", finished.is_checkmate())
print("Result:", finished.result())

stalemate = chess.Board("7k/5Q2/6K1/8/8/8/8/8 b - - 0 1")
print()
print("Black to move. Stalemate?", stalemate.is_stalemate())
print("Result:", stalemate.result())
stalemate

# Writing a chess player

A chess-playing program can be a function that takes a board and returns a move. The simplest one picks any legal move at random.

`play_game()` asks each player for a move in turn until the game is over. Run the cell a few times: random players make some strange moves, and most of their games end in a draw.

In [ ]:
#@alt Chess board at the end of a game between two random players
import random

def random_player(board):
  return random.choice(list(board.legal_moves))

def play_game(white, black):
  board = chess.Board()
  while not board.is_game_over():
    player = white if board.turn == chess.WHITE else black
    board.push(player(board))
  return board

game = play_game(random_player, random_player)
print("Result:", game.result())
print("Ended by:", game.outcome().termination.name)
print("Moves played:", len(game.move_stack))
game

## A smarter player

This player follows two rules:

1. If a move gives checkmate, play it. It finds one with the try, check, undo pattern from the puzzle.
2. Otherwise, capture the most valuable piece it can, using the usual piece values: pawn 1, knight and bishop 3, rook 5, queen 9.

When nothing can be captured, every move scores 0 and it picks any of them. Shuffling the moves first means it doesn't always pick the same one.

In [ ]:
PIECE_VALUES = {
  chess.PAWN: 1,
  chess.KNIGHT: 3,
  chess.BISHOP: 3,
  chess.ROOK: 5,
  chess.QUEEN: 9,
  chess.KING: 0,
}

def capture_value(board, move):
  if board.is_en_passant(move):
    return PIECE_VALUES[chess.PAWN]
  captured = board.piece_at(move.to_square)
  return PIECE_VALUES[captured.piece_type] if captured else 0

def greedy_player(board):
  moves = list(board.legal_moves)
  random.shuffle(moves)

  # Rule 1: checkmate if we can
  for move in moves:
    board.push(move)
    mate = board.is_checkmate()
    board.pop()
    if mate:
      return move

  # Rule 2: take the most valuable piece on offer
  return max(moves, key=lambda move: capture_value(board, move))

print("greedy_player is ready")

Now let the two players play a match. They swap colours every game, so neither always moves first. How much better do two simple rules make the greedy player?

In [ ]:
#@title Greedy vs. random
GAMES = 10 #@param {type:"slider", min:2, max:30, step:2}

results = {"Greedy wins": 0, "Random wins": 0, "Draws": 0}

for number in range(GAMES):
  if number % 2 == 0:
    game = play_game(greedy_player, random_player)
    greedy_colour = chess.WHITE
  else:
    game = play_game(random_player, greedy_player)
    greedy_colour = chess.BLACK

  winner = game.outcome().winner  # chess.WHITE, chess.BLACK, or None for a draw
  if winner is None:
    results["Draws"] += 1
  elif winner == greedy_colour:
    results["Greedy wins"] += 1
  else:
    results["Random wins"] += 1

for name, count in results.items():
  print(f"{name}: {count}")

# Things to Try

- Add a rule to `greedy_player` that prefers moves giving check. `board.gives_check(move)` tells you whether a move does.
- Make `greedy_player` more careful: after each capture it considers, check whether the opponent could capture back, and prefer moves that don't lose a piece.
- Find a "mate in one" puzzle on a chess website, copy its FEN into `chess.Board()`, and let the search find the answer.
- Write a `human_player(board)` that asks for your move with `input()` and turns it into a move with `board.parse_san()`. Then play against `greedy_player`.

# Check Your Understanding